<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/03a-vanishing-exploding-gradients.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import model_selection, pipeline, preprocessing
import torch
from torch import nn, optim, utils

# Vanishing and Exploding Gradients

**Vanishing gradients**: gradients get smaller and smaller until parameters in early layers get updates so small that the model effectively stops learning. When this happens the training process fails to converge to a good solution.

**Exploding gradients**: gradients get bigger and bigger until the paramters get updates so large that the training process begins to diverge!



### Define some utility functions

The code in the cell below defines a few utility functions that will make our life easier.

In [ ]:
def compute_average_loss_per_batch(dataloader, criterion, model_fn):
    total_loss = torch.zeros(1, 1)
    num_batches = len(dataloader)
    for features, targets in dataloader:
        predictions = model_fn(features)
        batch_loss = criterion(predictions, targets)
        total_loss += batch_loss
    average_loss_per_batch = total_loss / num_batches
    return average_loss_per_batch


def fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device="cpu",
    log_epochs=1,
    max_epochs=1
    ):

    history = {
        "epoch": [],
        "average_train_loss": [],
        "average_val_loss": []
    }
    num_train_batches = len(train_dataloader)
    for epoch in range(max_epochs):
        total_train_loss = torch.zeros(1, 1)
        model_fn = model_fn.train()
        for features, targets in train_dataloader:

            # forward pass
            features, targets = features.to(device), targets.to(device)
            predictions = model_fn(features)
            batch_loss = criterion(predictions, targets)
            total_train_loss += batch_loss

            # backward pass
            optimizer.zero_grad()
            batch_loss.backward()
            optimizer.step()

        history["epoch"].append(epoch)

        average_train_loss_per_batch = total_train_loss / num_train_batches
        history["average_train_loss"].append(average_train_loss_per_batch.item())

        model_fn = model_fn.eval()
        with torch.inference_mode():
            average_val_loss_per_batch = compute_average_loss_per_batch(
                val_dataloader,
                criterion,
                model_fn
            )
        history["average_val_loss"].append(average_val_loss_per_batch.item())


        if (epoch + 1) % log_epochs == 0:
            print(
                f"Epoch {epoch},",
                f"Average train Loss {average_train_loss_per_batch.item():.4f},",
                f"Average val Loss {average_val_loss_per_batch.item():.4f}"
            )

    history_df = (
        pd.DataFrame.from_dict(history)
                    .set_index("epoch")
    )

    return history_df


## Load the MNIST data

In [ ]:
%%bash
cat ./sample_data/mnist_train_small.csv | head -n 5

In [ ]:
INPUT_SIZE = 784
OUTPUT_SIZE = 10
RANDOM_STATE = np.random.RandomState(42)


_train_data_df = pd.read_csv(
    "./sample_data/mnist_train_small.csv",
    header=None,
    names=["label"] + [f"p{i}" for i in range(INPUT_SIZE)],
)
train_data_df, val_data_df = model_selection.train_test_split(
    _train_data_df,
    random_state=RANDOM_STATE,
    stratify=_train_data_df.loc[:, "label"],
    test_size=0.1,
)

test_data_df = pd.read_csv(
    "./sample_data/mnist_test.csv",
    header=None,
    names=["label"] + [f"p{i}" for i in range(INPUT_SIZE)],
)

### Create preprocessing pipelines

In [ ]:
def array_to_tensor(arr, dtype=torch.float32):
  return torch.tensor(arr, dtype=dtype)


def series_to_tensor(s, dtype=torch.float32):
    arr = s.to_numpy()
    return array_to_tensor(arr, dtype)


features_preprocessor = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    preprocessing.FunctionTransformer(
        array_to_tensor,
        kw_args={
            "dtype":
            torch.float32
        }
    ),
)

target_preprocessor = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        series_to_tensor,
        kw_args={
            "dtype":
            torch.int64
        }
    ),
)


### Create Datasets and DataLoaders

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2


# create the training dataset and dataloader
train_features_tensor = features_preprocessor.fit_transform(
    train_data_df.drop("label", axis=1)
)

train_target_tensor = target_preprocessor.fit_transform(
    train_data_df.loc[:, "label"]
)

train_dataset = utils.data.TensorDataset(
    train_features_tensor,
    train_target_tensor
)

train_dataloader = utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    persistent_workers=True,
    num_workers=NUM_WORKERS,
    shuffle=True,
)

# create the validation dataset and dataloader
val_features_tensor = features_preprocessor.transform(
    val_data_df.drop("label", axis=1)
)

val_target_tensor = target_preprocessor.transform(
    val_data_df.loc[:, "label"]
)

val_dataset = utils.data.TensorDataset(
    val_features_tensor,
    val_target_tensor
)

val_dataloader = utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    persistent_workers=True,
    num_workers=NUM_WORKERS,
    shuffle=False
)

# create the test dataset and dataloader
test_features_tensor = features_preprocessor.transform(
    test_data_df.drop("label", axis=1)
)

test_target_tensor = target_preprocessor.transform(
    test_data_df.loc[:, "label"]
)

test_dataset = utils.data.TensorDataset(
    test_features_tensor,
    test_target_tensor
)

test_dataloader = utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    persistent_workers=False,
    num_workers=NUM_WORKERS,
    shuffle=False
)


## Vanishing gradients example

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HIDDEN_SIZE = int((2 / 3) * (INPUT_SIZE + OUTPUT_SIZE))
LEARNING_RATE = 1e-3
MAX_EPOCHS = 10

loss_fn = nn.CrossEntropyLoss()

model_fn = nn.Sequential(
    nn.Linear(INPUT_SIZE, HIDDEN_SIZE),
    nn.Sigmoid(),
    nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
    nn.Sigmoid(),
    nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
    nn.Sigmoid(),
    nn.Linear(HIDDEN_SIZE, OUTPUT_SIZE),
)
model_fn = model_fn.to(DEVICE)


optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

history_df = fit(
    loss_fn,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

Notice how the both train and val losses fail to decrease: the Sigmoid activation function has become saturated and as such the gradient is basically zero: parameters are not updating, model is not learning, so training is not making any progress.

## Better Activation Functions

In this section we will explore how different activation functions can address the problem of vanishing gradients.

The code in the cell below defines another utility function to help us generate MLP classifiers with different activation functions.

In [ ]:
def make_mlp_classifier(
    input_size,
    hidden_sizes=None,
    output_size=2,
    activation_fn=None
    ):
    modules = []
    hidden_sizes = [] if hidden_sizes is None else hidden_sizes
    for hidden_size in hidden_sizes:
        hidden_layer = nn.Linear(input_size, hidden_size)
        modules.append(hidden_layer)
        if activation_fn is not None:
            modules.append(activation_fn)
        input_size=hidden_size
    output_layer = nn.Linear(input_size, output_size)
    modules.append(output_layer)
    model_fn = nn.Sequential(*modules)
    return nn.CrossEntropyLoss(), model_fn


### ReLU

* ReLU does not suffer from vanishing gradients for positive values and is very fast to compute.

* ReLU can suffer from the problem of "dying" ReLUs if too many neurons output negative values as the ReLU will then output zero.

In [ ]:
nn.ReLU?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.ReLU()
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(), lr=LEARNING_RATE)

relu_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = relu_history_df.plot(grid=True, ylim=(0, 2.5))

### Leaky ReLU

* Hyperparameter controls how much the activation function "leaks".
* Having non-zero slope for negative ouputs solves the "dying ReLUs" problem.

In [ ]:
nn.LeakyReLU?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.LeakyReLU(negative_slope=0.01)
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

leaky_relu_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = leaky_relu_history_df.plot(grid=True, ylim=(0, 2.5))

### Exercise:

There are a couple of variants of Leaky ReLU: Randomized Leaky ReLU and Parametric Leaky ReLU. Adapt the code above to train models using these variants. Plot the training and validation losses and discuss.

In [ ]:
nn.RReLU?

In [ ]:
nn.PReLU?

#### Solution:

### ELU and SELU

**Exponential Linear Unit (ELU)** is *negative* when neuron outputs a negative number. Has a hyperparameter that determines the value that the function approahes when neuron outputs are large and negative.

**Scaled ELU (SELU)** often used when training MLPs as this activation function allows the network to self-normalize!

In [ ]:
nn.ELU?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.ELU(alpha=1.0)
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

elu_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = elu_history_df.plot(grid=True, ylim=(0, 2.5))

In [ ]:
nn.SELU?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU()
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

selu_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = selu_history_df.plot(grid=True, ylim=(0, 2.5))

### GELU, Swish, Mish

**Gaussian Error Linear Unit (GELU)**, **Sigmoid Linear Unit (SiLU or Swish)**, and **Mish** are all smooth, non-monotonic, non-convex, ReLU variants.

The idea with these activation functions is that while the extra complexity of the functions (relative to ReLU) takes more compute time during training (and inference), the training process will converge to a good solution in fewer iterations.

In [ ]:
nn.GELU?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.GELU(approximate="none")
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

gelu_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = gelu_history_df.plot(grid=True, ylim=(0, 2.5))

In [ ]:
nn.SiLU?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SiLU()
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

silu_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = silu_history_df.plot(grid=True, ylim=(0, 2.5))

In [ ]:
nn.Mish?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.Mish()
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

mish_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
_ = mish_history_df.plot(grid=True, ylim=(0, 2.5))

### Exercise:

Compare and contrast the plots of the training and validation loss curves for the difference activation functions.

#### Solution

## Better Parameter Initialization Strategies

With deeper models that have more parameters, choosing the right parameter initialization strategy can be critical.

In [ ]:
def initialize_linear_layer(
    in_features,
    out_features,
    init_strategy_=nn.init.kaiming_uniform_,
    init_strategy_kwargs=None,
    ):
    linear_layer = nn.Linear(in_features, out_features)

    if init_strategy_ is not None:
        if init_strategy_kwargs is None:
            init_strategy_kwargs = {}
        init_strategy_(linear_layer.weight, **init_strategy_kwargs)
        linear_layer.bias.data.fill_(0.0)

    return linear_layer


In [ ]:
def make_mlp_classifier(
    input_size,
    hidden_sizes=None,
    output_size=2,
    activation_fn=None,
    init_strategy_=nn.init.kaiming_uniform_,
    init_strategy_kwargs=None,
    ):
    modules = []
    hidden_sizes = [] if hidden_sizes is None else hidden_sizes
    for hidden_size in hidden_sizes:
        hidden_layer = initialize_linear_layer(
            input_size,
            hidden_size,
            init_strategy_,
            init_strategy_kwargs,
        )
        modules.append(hidden_layer)
        if activation_fn is not None:
            modules.append(activation_fn)
        input_size=hidden_size
    output_layer = initialize_linear_layer(
            input_size,
            output_size,
            init_strategy_,
            init_strategy_kwargs,
    )
    modules.append(output_layer)
    model_fn = nn.Sequential(*modules)
    return nn.CrossEntropyLoss(), model_fn

### Kaiming

In [ ]:
nn.init.kaiming_normal_?

In [ ]:
nn.init.kaiming_uniform_?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.ReLU(),
    init_strategy_=nn.init.kaiming_uniform_
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

relu_kaiming_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

_ = relu_history_df.plot(ax=axes[0], grid=True, title="ReLU with Default Initialization")
_ = relu_kaiming_history_df.plot(ax=axes[1], grid=True, title="ReLU with Kaiming (Uniform) Initialization")

### Xavier

In [ ]:
nn.init.xavier_normal_?

In [ ]:
nn.init.xavier_uniform_?

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.ReLU(),
    init_strategy_=nn.init.xavier_normal_
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

relu_xavier_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

_ = relu_kaiming_history_df.plot(ax=axes[0], grid=True, title="Kaiming (Uniform)")
_ = relu_xavier_history_df.plot(ax=axes[1], grid=True, title="Xavier (Normal)")

### Exercise:

Choose the appropriate activation function and initialization strategy to implement a self-normalizing MLP. Your MLP should have three hidden layers, each layer with `HIDDEN_SIZE` neurons per layer. Train your MLP for 10 epochs on the MNIST dataset using Stochastic Gradient Descent with a learning rate of 1e-3. Plot the training and validation loss curves.

#### Solution:

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SELU(),
    init_strategy_=nn.init.kaiming_normal_,
    init_strategy_kwargs={
        "mode": "fan_in",
        "nonlinearity": "linear"
    }
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

selu_kaiming_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

_ = relu_history_df.plot(ax=axes[0], grid=True, title="ReLU with Default Initialization")
_ = selu_kaiming_history_df.plot(ax=axes[1], grid=True, title="Self-Normalizing MLP")

## Batch Normalization

In [ ]:
nn.BatchNorm1d?

In [ ]:
def make_mlp_classifier(
    input_size,
    hidden_sizes=None,
    output_size=2,
    activation_fn=None,
    init_strategy_=nn.init.kaiming_uniform_,
    init_strategy_kwargs=None,
    batch_normalization=False
    ):
    modules = []
    hidden_sizes = [] if hidden_sizes is None else hidden_sizes
    for hidden_size in hidden_sizes:
        hidden_layer = initialize_linear_layer(
            input_size,
            hidden_size,
            init_strategy_,
            init_strategy_kwargs,
        )
        modules.append(hidden_layer)

        # batch normalization goes after the linear layer...
        if batch_normalization:
            modules.append(nn.BatchNorm1d(hidden_size))

        # ...but before the activation_fn!
        if activation_fn is not None:
            modules.append(activation_fn)
        input_size=hidden_size
    output_layer = initialize_linear_layer(
            input_size,
            output_size,
            init_strategy_,
            init_strategy_kwargs,
    )
    modules.append(output_layer)
    model_fn = nn.Sequential(*modules)
    return nn.CrossEntropyLoss(), model_fn

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.ReLU(),
    init_strategy_=nn.init.kaiming_uniform_,
    batch_normalization=True
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

relu_kaiming_batch_norm_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

_ = relu_kaiming_history_df.plot(ax=axes[0], grid=True, title="ReLU with Kaiming (Uniform) Initialization")
_ = relu_kaiming_batch_norm_history_df.plot(ax=axes[1], grid=True, title="ReLU, Kaiming (Uniform), Batch Normalization")

### Exercise:

Create an MLP with three hidden layers, each layer with `HIDDEN_SIZE` neurons per layer, SiLU activation function, and the `kaiming_normal` initialization strategy. Train your MLP for 10 epochs on the MNIST dataset using Stochastic Gradient Descent with a learning rate of 1e-3. Plot the training and validation loss curves.

#### Solution:

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SiLU(),
    init_strategy_=nn.init.kaiming_normal_,
    batch_normalization=False
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

silu_kaiming_normal_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.SiLU(),
    init_strategy_=nn.init.kaiming_normal_,
    batch_normalization=True
)
model_fn = model_fn.to(DEVICE)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

silu_kaiming_normal_batch_norm_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    max_epochs=MAX_EPOCHS
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

_ = silu_kaiming_normal_history_df.plot(ax=axes[0], grid=True, title="Swish with Kaiming (Normal) Initialization")
_ = silu_kaiming_normal_batch_norm_history_df.plot(ax=axes[1], grid=True, title="Swish, Kaiming (Normal), Batch Normalization")

## Gradient Clipping

In [ ]:
nn.utils.clip_grad_value_?

In [ ]:
nn.utils.clip_grad_norm_?

In [ ]:
def clip_gradients_(
    clip_grad_strategy,
    model_fn,
    clip_value=None,
    error_if_nonfinite=False,
    max_norm=None,
    norm_type=2.0):
    if clip_grad_strategy == "value" and clip_value is not None:
        nn.utils.clip_grad_value_(
            model_fn.parameters(),
            clip_value
        )
    elif clip_grad_strategy == "norm" and max_norm is not None:
        nn.utils.clip_grad_norm_(
            model_fn.parameters(),
            max_norm,
            norm_type,
            error_if_nonfinite
        )
    else:
        raise NotImplementedError()

def fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device="cpu",
    clip_grad_strategy=None,
    clip_value=None,
    error_if_nonfinite=False,
    log_epochs=1,
    max_epochs=1,
    max_norm=None,
    norm_type=2.0
    ):

    history = {
        "epoch": [],
        "average_train_loss": [],
        "average_val_loss": []
    }
    num_train_batches = len(train_dataloader)
    for epoch in range(max_epochs):
        total_train_loss = torch.zeros(1, 1)
        model_fn = model_fn.train()
        for features, targets in train_dataloader:

            # forward pass
            features, targets = features.to(device), targets.to(device)
            predictions = model_fn(features)
            batch_loss = criterion(predictions, targets)
            total_train_loss += batch_loss

            # backward pass
            optimizer.zero_grad()
            batch_loss.backward()
            clip_gradients_(
                clip_grad_strategy,
                model_fn,
                clip_value,
                error_if_nonfinite,
                max_norm,
                norm_type
            )
            optimizer.step()

        history["epoch"].append(epoch)

        average_train_loss_per_batch = total_train_loss / num_train_batches
        history["average_train_loss"].append(average_train_loss_per_batch.item())

        model_fn = model_fn.eval()
        with torch.inference_mode():
            average_val_loss_per_batch = compute_average_loss_per_batch(
                val_dataloader,
                criterion,
                model_fn
            )
        history["average_val_loss"].append(average_val_loss_per_batch.item())


        if (epoch + 1) % log_epochs == 0:
            print(
                f"Epoch {epoch},",
                f"Average train Loss {average_train_loss_per_batch.item():.4f},",
                f"Average val Loss {average_val_loss_per_batch.item():.4f}"
            )

    history_df = (
        pd.DataFrame.from_dict(history)
                    .set_index("epoch")
    )

    return history_df


In [ ]:
criterion, model_fn = make_mlp_classifier(
    input_size=INPUT_SIZE,
    hidden_sizes=[HIDDEN_SIZE, HIDDEN_SIZE, HIDDEN_SIZE],
    output_size=OUTPUT_SIZE,
    activation_fn=nn.ReLU(),
    init_strategy_=None,
    init_strategy_kwargs=None,
    batch_normalization=False,
)

optimizer = optim.SGD(
    model_fn.parameters(),
    lr=LEARNING_RATE
)

relu_clipped_grads_history_df = fit(
    criterion,
    model_fn,
    optimizer,
    train_dataloader,
    val_dataloader,
    device=DEVICE,
    clip_grad_strategy="norm",
    max_norm=1.0,
    error_if_nonfinite=True,
    max_epochs=MAX_EPOCHS
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

_ = relu_history_df.plot(ax=axes[0], grid=True, title="ReLU")
_ = relu_clipped_grads_history_df.plot(ax=axes[1], grid=True, title="ReLU with Gradient Clipping")


### Exercise:

Clipping gradients does not appear to make much of a difference. Why?

#### Solution: